In [ ]:
import os

# Move one directory back
os.chdir('..')

# Verify the current working directory
print("Current working directory:", os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter
import biosppy
from scipy.fftpack import fft

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import biosppy
from biosppy.signals import ppg
from biosppy.signals.tools import filter_signal
import plotly.graph_objs as go
from scipy.signal import find_peaks, butter, filtfilt
import librosa
import neurokit2 as nk

from ppg_preprocess import generate_synthetic_ppg, add_noise, process_ppg_with_biosppy, process_ppg,is_ppg_signal 
from scipy.signal import butter, filtfilt
from ppg_preprocess import butter_bandpass, bandpass_filter,plot_fft

# EXTRACT SAMPLE DATA AND EXPORT

In [ ]:
import pandas as pd

# Load your data
df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/adhd_train/adhd-KKI_001-right-sync.csv')
#df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/test/california-00000192-right-sync.csv')

# Create sleep_label: 0 for awake, 1 for sleep
df['sleep_label'] = df['sleep_stage'].apply(lambda x: 0 if x == 'WK' else 1)

# Drop rows with missing values in critical columns
#df = df.dropna(subset=['ledIR', 'ledRed', 'ledGreen', 'sleep_label'])
#df = df.dropna(subset=['accelerometerX', 'accelerometerY', 'accelerometerZ', 'sleep_label'])
df = df.dropna(subset=['gyroscopeX', 'gyroscopeY', 'gyroscopeZ', 'sleep_label'])

# Convert Unix time to human-readable format
df['formatted_time'] = pd.to_datetime(df['unixTimes'], unit='ms')


# Define start and end times for a 20-minute segment
start_time = df['formatted_time'].iloc[10]  # Starting from the beginning, adjust if needed
end_time = start_time + pd.Timedelta(minutes=20)

# Filter the DataFrame for the 20-minute segment
segment_df = df[(df['formatted_time'] >= start_time) & (df['formatted_time'] <= end_time)]

# Select relevant columns: Unix time, filtered PPG signal, and sleep stage
export_df = segment_df[['unixTimes', 'gyroscopeX', 'gyroscopeY', 'gyroscopeZ', 'sleep_label']]
#export_df = segment_df[['unixTimes', 'accelerometerX', 'accelerometerY', 'accelerometerZ', 'sleep_label']]

# Export the segment to a CSV file
export_df.to_csv('adhd-KKI_001-right_gyro_20_minute_segment.csv', index=False)

print("20-minute segment successfully exported to 'ppg_20_minute_segment.csv'")


In [ ]:
df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/adhd_train/adhd-KKI_001-right-sync.csv')
#df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/test/california-00000192-right-sync.csv')

df['sleep_label'] = df['sleep_stage'].apply(lambda x: 0 if x == 'WK' else 1)  # 0: awake, 1: sleep
df = df.dropna(subset=['ledIR', 'ledRed', 'ledGreen', 'sleep_label'])
df['formatted_time'] = pd.to_datetime(df['unixTimes'], unit='ms')

fs = 25
lowcut = 0.2
highcut = 5.0

df['ledGreen_filtered']=bandpass_filter(df['ledGreen'].dropna(), lowcut, highcut, fs)
df['ledRed_filtered']=bandpass_filter(df['ledRed'].dropna(), lowcut, highcut, fs)
df['ledIR_filtered']=bandpass_filter(df['ledIR'].dropna(), lowcut, highcut, fs)

In [ ]:
ppg_results = biosppy.signals.ppg.ppg(signal=df['ledGreen'].values, sampling_rate=25, show=True)

# Extract the filtered signal and heart rate
heart_rate = ppg_results['heart_rate']
print("Average heart_rate :", np.mean(heart_rate))
print("heart_rate",heart_rate)

In [ ]:
# Create Plotly figure
fig = go.Figure()

# Add trace for processed ledGreen signal
fig.add_trace(go.Scatter(x=df['formatted_time'], y=df['ledGreen_filtered'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(x=df['formatted_time'], y=df['ledGreen'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(
    x=df[df['sleep_label'] == 1]['formatted_time'], 
    y=df[df['sleep_label'] == 1]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='blue', dash='dot'),  # Blue dashed line for sleep periods
    name='Sleep Periods (Label 1)'
))

# Highlight wake periods (sleep_label == 0) with another color
fig.add_trace(go.Scatter(
    x=df[df['sleep_label'] == 0]['formatted_time'], 
    y=df[df['sleep_label'] == 0]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='orange', dash='dash'),  # Orange dashed line for wake periods
    name='Wake Periods (Label 0)'
))
# Update layout
fig.update_layout(title='Processed ledGreen Signal with Sleep Periods',
                  xaxis_title='Time (unixTimes)',
                  yaxis_title='Signal Value',
                  hovermode='x unified')

# Show the plot
fig.show()


# example 2

In [ ]:
df2 = pd.read_csv('/home/yanwar/Desktop/Sleep/data/adhd_train/adhd-KKI_003-left-sync.csv')
#df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/test/california-00000192-right-sync.csv')

df2['sleep_label'] = df2['sleep_stage'].apply(lambda x: 0 if x == 'WK' else 1)  # 0: awake, 1: sleep
df2 = df2.dropna(subset=['ledIR', 'ledRed', 'ledGreen', 'sleep_label'])
df2['formatted_time'] = pd.to_datetime(df2['unixTimes'], unit='ms')

fs = 25
lowcut = 0.2
highcut = 5.0

df2['ledGreen_filtered']=bandpass_filter(df2['ledGreen'].dropna(), lowcut, highcut, fs)
df2['ledRed_filtered']=bandpass_filter(df2['ledRed'].dropna(), lowcut, highcut, fs)
df2['ledIR_filtered']=bandpass_filter(df2['ledIR'].dropna(), lowcut, highcut, fs)

In [ ]:
ppg_results = biosppy.signals.ppg.ppg(signal=df2['ledGreen'].values, sampling_rate=25, show=True)

# Extract the filtered signal and heart rate
heart_rate = ppg_results['heart_rate']
print("Average heart_rate :", np.mean(heart_rate))
print("heart_rate",heart_rate)

In [ ]:
# Create Plotly figure
fig = go.Figure()

# Add trace for processed ledGreen signal
fig.add_trace(go.Scatter(x=df2['formatted_time'], y=df2['ledGreen_filtered'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(x=df2['formatted_time'], y=df2['ledGreen'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(
    x=df2[df2['sleep_label'] == 1]['formatted_time'], 
    y=df2[df2['sleep_label'] == 1]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='blue', dash='dot'),  # Blue dashed line for sleep periods
    name='Sleep Periods (Label 1)'
))

# Highlight wake periods (sleep_label == 0) with another color
fig.add_trace(go.Scatter(
    x=df2[df2['sleep_label'] == 0]['formatted_time'], 
    y=df2[df2['sleep_label'] == 0]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='orange', dash='dash'),  # Orange dashed line for wake periods
    name='Wake Periods (Label 0)'
))
# Update layout
fig.update_layout(title='Processed ledGreen Signal with Sleep Periods',
                  xaxis_title='Time (unixTimes)',
                  yaxis_title='Signal Value',
                  hovermode='x unified')

# Show the plot
fig.show()


# example 3

In [ ]:
df3 = pd.read_csv('/home/yanwar/Desktop/Sleep/data/adhd_train/adhd-KKI_011-right-sync.csv')
#df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/test/california-00000192-right-sync.csv')

df3['sleep_label'] = df3['sleep_stage'].apply(lambda x: 0 if x == 'WK' else 1)  # 0: awake, 1: sleep
df3 = df3.dropna(subset=['ledIR', 'ledRed', 'ledGreen', 'sleep_label'])
df3['formatted_time'] = pd.to_datetime(df3['unixTimes'], unit='ms')

fs = 25
lowcut = 0.2
highcut = 5.0

df3['ledGreen_filtered']=bandpass_filter(df3['ledGreen'].dropna(), lowcut, highcut, fs)
df3['ledRed_filtered']=bandpass_filter(df3['ledRed'].dropna(), lowcut, highcut, fs)
df3['ledIR_filtered']=bandpass_filter(df3['ledIR'].dropna(), lowcut, highcut, fs)

In [ ]:
ppg_results = biosppy.signals.ppg.ppg(signal=df3['ledGreen'].values, sampling_rate=25, show=True)

# Extract the filtered signal and heart rate
heart_rate = ppg_results['heart_rate']
print("Average heart_rate :", np.mean(heart_rate))
print("heart_rate",heart_rate)

In [ ]:
# Create Plotly figure
fig = go.Figure()

# Add trace for processed ledGreen signal
fig.add_trace(go.Scatter(x=df3['formatted_time'], y=df3['ledGreen_filtered'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(x=df3['formatted_time'], y=df3['ledGreen'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(
    x=df2[df2['sleep_label'] == 1]['formatted_time'], 
    y=df2[df2['sleep_label'] == 1]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='blue', dash='dot'),  # Blue dashed line for sleep periods
    name='Sleep Periods (Label 1)'
))

# Highlight wake periods (sleep_label == 0) with another color
fig.add_trace(go.Scatter(
    x=df2[df2['sleep_label'] == 0]['formatted_time'], 
    y=df2[df2['sleep_label'] == 0]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='orange', dash='dash'),  # Orange dashed line for wake periods
    name='Wake Periods (Label 0)'
))
# Update layout
fig.update_layout(title='Processed ledGreen Signal with Sleep Periods',
                  xaxis_title='Time (unixTimes)',
                  yaxis_title='Signal Value',
                  hovermode='x unified')

# Show the plot
fig.show()


In [ ]:
# Set the time gap threshold (e.g., 2 minutes = 120 seconds)
time_gap_threshold = 120  # in seconds

# Function to identify and plot large time gaps
def identify_and_plot_time_gaps(df, filename):
    # Convert Unix timestamps to datetime
    df['datetime'] = pd.to_datetime(df['unixTimes'], unit='ms', errors='coerce')
    
    # Sort by datetime to ensure chronological order
    df = df.sort_values(by='datetime')
    
    # Calculate the time difference between consecutive rows (in seconds)
    df['time_diff'] = df['datetime'].diff().dt.total_seconds()
    
    # Identify the rows where the time gap exceeds the threshold
    large_gaps = df[df['time_diff'] > time_gap_threshold]

    # Plot the time series and highlight the large gaps
    plt.figure(figsize=(12, 6))
    
    # Plot the Unix times (converted to datetime)
    plt.plot(df['datetime'], df['unixTimes'], label='Unix Time', color='blue')
    
    # Highlight the rows with large time gaps
    if not large_gaps.empty:
        plt.scatter(large_gaps['datetime'], large_gaps['unixTimes'], color='red', label=f'Gaps > {time_gap_threshold} sec', zorder=5)
    
    plt.xlabel('Time')
    plt.ylabel('Unix Time')
    plt.title(f'Time Series with Large Gaps - {filename}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    # Save the plot for each file
    #plt.savefig(f'plots/gaps/{filename}_time_gaps.png')
    #plt.close()

    return large_gaps


In [ ]:

folder_path = '/home/yanwar/Desktop/Sleep/data/adhd_train/'
#folder_path = '/home/yanwar/Desktop/Sleep/data/test/'

# Iterate over all CSV files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # Analyze and plot large time gaps for each file
        large_gaps = identify_and_plot_time_gaps(df, filename)

        if not large_gaps.empty:
            print(f'Large time gaps identified in file {filename}:')
            print(large_gaps[['unixTimes', 'datetime', 'time_diff']])
    

In [ ]:
# Function to identify and plot large time gaps with index
def identify_and_plot_time_gaps(df, filename):
    # Convert Unix timestamps to datetime
    df['datetime'] = pd.to_datetime(df['unixTimes'], unit='ms', errors='coerce')
    
    # Sort by datetime to ensure chronological order
    df = df.sort_values(by='datetime').reset_index(drop=True)
    
    # Calculate the time difference between consecutive rows (in seconds)
    df['time_diff'] = df['datetime'].diff().dt.total_seconds()
    
    # Plot the index vs. datetime to visualize time jumps
    plt.figure(figsize=(12, 6))
    
    # Plot index on x-axis and datetime on y-axis
    plt.plot(df.index, df['datetime'], label='Datetime', color='blue', marker='o', markersize=2)

    # Highlight the rows with large time gaps
    large_gaps = df[df['time_diff'] > time_gap_threshold]
    
    if not large_gaps.empty:
        plt.scatter(large_gaps.index, large_gaps['datetime'], color='red', label=f'Gaps > {time_gap_threshold} sec', zorder=5)
    
    plt.xlabel('Data Point Index (Number of Data Points)')
    plt.ylabel('Datetime (Converted from Unix Time)')
    plt.title(f'Index vs Datetime - Time Gaps in {filename}')
    plt.legend()
    plt.tight_layout()
    
    # Save the plot for each file
    #plt.savefig(f'plots/gaps/{filename}_index_vs_time_gaps.png')
    plt.show()
    
    return large_gaps

In [ ]:

folder_path = '/home/yanwar/Desktop/Sleep/data/adhd_train/'
#folder_path = '/home/yanwar/Desktop/Sleep/data/test/'

# Iterate over all CSV files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # Analyze and plot large time gaps for each file
        large_gaps = identify_and_plot_time_gaps(df, filename)

        if not large_gaps.empty:
            print(f'Large time gaps identified in file {filename}:')
            print(large_gaps[['unixTimes', 'datetime', 'time_diff']])
    

In [ ]:

folder_path = '/home/yanwar/Desktop/Sleep/data/adhd_cleaned/'
#folder_path = '/home/yanwar/Desktop/Sleep/data/test/'

# Iterate over all CSV files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # Analyze and plot large time gaps for each file
        large_gaps = identify_and_plot_time_gaps(df, filename)

        if not large_gaps.empty:
            print(f'Large time gaps identified in file {filename}:')
            print(large_gaps[['unixTimes', 'datetime', 'time_diff']])
    

In [ ]:
# Define the folder paths and time range
folder_path = '/home/yanwar/Desktop/Sleep/data/adhd_train/'
output_folder_path = '/home/yanwar/Desktop/Sleep/data/adhd_cleaned/'

# Make sure the output folder exists
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

# Define the valid time range (2023-2024) in Unix milliseconds
start_time_a = pd.Timestamp('2023-01-01').timestamp() * 1000  # Convert to Unix time in milliseconds
end_time_a = pd.Timestamp('2024-12-31').timestamp() * 1000    # Convert to Unix time in milliseconds

# Function to clean and retain largest continuous segment
def clean_and_export_continuous_segment(file_path, output_folder_path):
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Convert Unix time to datetime for easier filtering
    df['datetime'] = pd.to_datetime(df['unixTimes'], unit='ms', errors='coerce')

    # Filter out rows outside of the valid date range (2023-2024)
    df_cleaned = df[(df['unixTimes'] >= start_time) & (df['unixTimes'] <= end_time)]
    
    # Sort the dataframe by Unix time
    df_cleaned = df_cleaned.sort_values(by='unixTimes').reset_index(drop=True)

    # Calculate time differences between consecutive rows
    df_cleaned['time_diff'] = df_cleaned['unixTimes'].diff()

    # Identify segments where time difference is greater than 1 hour (in milliseconds)
    time_gap_threshold = 60 * 60 * 1000  # 1 hour in milliseconds
    large_gaps = df_cleaned['time_diff'] > time_gap_threshold

    # Label each segment based on the time gaps
    df_cleaned['segment'] = large_gaps.cumsum()

    # Find the largest continuous segment
    largest_segment = df_cleaned['segment'].value_counts().idxmax()
    largest_continuous_df = df_cleaned[df_cleaned['segment'] == largest_segment]

    # Drop the helper columns used for segmentation
    largest_continuous_df = largest_continuous_df.drop(columns=['datetime', 'time_diff', 'segment'])

    # Get the filename from the file path and create a new cleaned file path
    filename = os.path.basename(file_path)
    cleaned_file_path = os.path.join(output_folder_path, f'cleaned_{filename}')
    
    # Export the cleaned continuous segment to a new CSV file
    largest_continuous_df.to_csv(cleaned_file_path, index=False)
    
    print(f"Cleaned and continuous segment exported to: {cleaned_file_path}")



# Iterate over all CSV files in the folder and clean them
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        clean_and_export_continuous_segment(file_path, output_folder_path)